# Sentiment analysis 

## 1. Textblob-FR

Documentation: https://textblob.readthedocs.io/en/dev/

### Imports

In [1]:
from textblob import Blobber
from textblob_fr import PatternTagger, PatternAnalyzer

c:\Users\Ordinateur\Documents\tac_test\tac\.venv\Lib\site-packages\textblob\_text.py:218: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR1 = re.compile("^[A-Za-z]\.$")       # single letter, "T. De Smedt"
c:\Users\Ordinateur\Documents\tac_test\tac\.venv\Lib\site-packages\textblob\_text.py:219: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR2 = re.compile("^([A-Za-z]\.)+$")    # alternating letters, "U.S."
c:\Users\Ordinateur\Documents\tac_test\tac\.venv\Lib\site-packages\textblob_fr\_text.py:212: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR1 = re.compile("^[A-Za-z]\.$")       # single letter, "T. De Smedt"
c:\Users\Ordinateur\Documents\tac_test\tac\.venv\Lib\site-packages\textblob_fr\_text.py:213: SyntaxWarning: invalid escape sequence '\.'
  RE_ABBR2 = re.compile("^([A-Za-z]\.)+$")    # alternating letters, "U.S."


### Création d'une fonction `get_sentiment`

In [2]:
tb = Blobber(pos_tagger=PatternTagger(), analyzer=PatternAnalyzer())

def get_sentiment(input_text):
    blob = tb(input_text)
    polarity, subjectivity = blob.sentiment
    polarity_perc = f"{100*abs(polarity):.0f}"
    subjectivity_perc = f"{100*subjectivity:.0f}"
    if polarity > 0:
        polarity_str = f"{polarity_perc}% positive"
    elif polarity < 0:
        polarity_str = f"{polarity_perc}% negative"
    else:
        polarity_str = "neutral"
    if subjectivity > 0:
        subjectivity_str = f"{subjectivity_perc}% subjective"
    else:
        subjectivity_str = "perfectly objective"
    print(f"This text is {polarity_str} and {subjectivity_str}.")

### Analyser le sentiment d'une phrase

In [3]:
get_sentiment("Ce journal est vraiment super intéressant.")

This text is 65% positive and 75% subjective.


In [4]:
get_sentiment("Cette phrase est négative et je ne suis pas content !")

This text is 41% negative and 60% subjective.


In [6]:
import os, re

txt_path = "../../data/txt"
YEAR = 1923
fichiers = sorted(f for f in os.listdir(txt_path) if f.startswith(f"KB_JB838_{YEAR}-"))

def chercher(mot, max_resultats=5):
    trouvees = []
    for f in fichiers:
        texte = open(os.path.join(txt_path, f), encoding="utf-8").read()
        texte = re.sub(r"\s+", " ", texte)
        for phrase in re.split(r"(?<=[.!?]) ", texte):
            if mot.lower() in phrase.lower() and 40 < len(phrase) < 300:
                trouvees.append((f, phrase))
    return trouvees[:max_resultats]

for mot in ["Ruhr", "gouvernement", "grève", "paix"]:
    print(f"\n=== {mot} ===")
    for f, p in chercher(mot):
        print(f"[{f}] {p}")
for mot in ["situation", "Allemagne", "crise", "espoir", "heureux", "triste"]:
    print(f"\n=== {mot} ===")
    for f, p in chercher(mot):
        print(f"[{f}] {p}")


=== Ruhr ===
[KB_JB838_1923-01-21_01-00001.txt] Il traite des mesures que les Français ont déjà prises et vont prendre dans la Ruhr, ét il >îes condamne sans aucun ménagement.
[KB_JB838_1923-01-21_01-00001.txt] Lloyd George, comme on le sait, passe actuellement ses vacanoes en Espagne, et,, depuis qu’H a éorifc l’article en question, la gravité de Ta situation dans la Ruhr s’est manifestement accrue d’heure en "heure.
[KB_JB838_1923-02-05_01-00002.txt] (.E >0114 L’OCCUPATION DE LA RUHR Ils s’attaquent aux malades , Strasbourg, 4 février.
[KB_JB838_1923-02-05_01-00002.txt] Décidément, tout est à l’optimisme en oe qui concerne la situation dan» la Ruhr.
[KB_JB838_1923-02-05_01-00002.txt] Le « Matin » caractérise la situation comme «uit* « Le 31 janvier, il était encore sorti de : la Ruhr, à destination de l'Allemagne, 122 wa~ f on® emportant 17,518 tonne® de coke èt 29,350 obnes ae houille.

=== gouvernement ===
[KB_JB838_1923-01-21_01-00001.txt] ALLEMANDES; A ANVERSf Des < maisons de l

In [7]:
import pandas as pd

phrases = [
    "Il traite des mesures que les Français ont déjà prises et vont prendre dans la Ruhr, ét il >îes condamne sans aucun ménagement.",
    "Décidément, tout est à l’optimisme en oe qui concerne la situation dan» la Ruhr.",
    "Barrè- re aurait déclaré que le gouvernement italien aurait fait des démarches pour se rappro - chér du gouvernement anglais.",
    "Les gouvernements se mirent également d’accord pour déférer le jugement du litige & un tribunal arbitral International.",
    "Aussi, si nous cornptons les métallurgistes et lestubiers en grève, nous pouvons dire qu'il y a actuellement 18,000 ouvriers sans travail â Verviers.",
    "Dans cette dernière viHe, les ouvriers des mines Kaisers tuh le et T remania se sont mis en grève pour des raisons de salaire.",
    "Les ,1er, 2 et 3 février,- il h’est pas sorti une tpnne de coke,.ni,une, tonne dé houille à destination de l’Allemagne.",
    "von Rieffcr, a déclaré : « Nous sommes menacés par une crise, qui dépassera-les moments les plus critiques de la guerre.",
    "C’est triste qn’on puisse ainsi aimer et oublier...",
    "C’est le frisson inoubliable 1 d’une trame amoureuse, captivante, un dénouement heureux, le tout commenté par l’orchestre d’élite de l’Albertum, en fa1: une œuvre grandiose, dépassant en splendeur tous les opéras connus.",
]

lignes = []
for p in phrases:
    pol, sub = tb(p).sentiment
    lignes.append([p, round(pol, 2), round(sub, 2)])

df_sent = pd.DataFrame(lignes, columns=["phrase", "polarité", "subjectivité"])
df_sent

,phrase,polarité,subjectivité
0,Il traite des mesures que les Français ont déj...,0.05,-0.10
1,"Décidément, tout est à l’optimisme en oe qui c...",0.22,0.10
2,Barrè- re aurait déclaré que le gouvernement i...,0.04,0.00
3,Les gouvernements se mirent également d’accord...,0.28,0.17
4,"Aussi, si nous cornptons les métallurgistes et...",0.10,0.60
5,"Dans cette dernière viHe, les ouvriers des min...",-0.29,0.20
6,"Les ,1er, 2 et 3 février,- il h’est pas sorti ...",0.00,0.00
7,"von Rieffcr, a déclaré : « Nous sommes menacés...",0.00,0.98
8,C’est triste qn’on puisse ainsi aimer et oubli...,-0.30,0.60
9,C’est le frisson inoubliable 1 d’une trame amo...,0.48,0.53


## 2. Utilisation de transformers

Documentation: https://github.com/TheophileBlard/french-sentiment-analysis-with-bert

**!!** Si le code ne tourne pas sur votre machine, vous pouvez le tester directement sur Google Colab en utilisant [ce lien](https://colab.research.google.com/github/TheophileBlard/french-sentiment-analysis-with-bert/blob/master/colab/french_sentiment_analysis_with_bert.ipynb) **!!**

Le modèle peut également être testé en ligne sur [HuggingFace](https://huggingface.co/tblard/tf-allocine)

Note : `transformers` v5 a supprimé le support TensorFlow. On utilise donc ici `philschmid/pt-tblard-tf-allocine`, un portage PyTorch du même modèle, avec les classes `AutoModelForSequenceClassification` (au lieu de `TFAutoModelForSequenceClassification`).

### Installation des librairies et imports

In [ ]:
!uv pip install sentencepiece

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

### Chargement du modèle

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("philschmid/pt-tblard-tf-allocine")
model = AutoModelForSequenceClassification.from_pretrained("philschmid/pt-tblard-tf-allocine")

sentiment_analyser = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

### Analyser le sentiment d'une phrase

In [ ]:
sentiment_analyser("Ce journal est vraiment super intéressant.")

In [ ]:
sentiment_analyser("Cette phrase est négative et je ne suis pas content !")